# 07. 10초 Segment 위치 기록

오디오 파일을 잘라 저장하는 대신 각 곡에서 사용할 시작·종료 시각을 표로 만든다. 30초 이상은 시작·가운데·끝, 20~30초는 시작·끝, 10~20초는 가운데 구간을 쓴다. 실제 길이가 30초에서 ±0.1초면 30초로 취급한다.

In [22]:
from pathlib import Path
import subprocess
import shutil
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("/Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project")

MASTER_PATH = PROJECT_ROOT / "data/metadata/master_manifest_with_split.csv"
DURATION_CACHE_PATH = PROJECT_ROOT / "data/metadata/audio_duration_cache.csv"
SEGMENT_PATH = PROJECT_ROOT / "data/metadata/segment_manifest_10s.csv"
EXCLUSION_PATH = PROJECT_ROOT / "data/metadata/segment_exclusions.csv"

SEGMENT_SEC = 10.0
NEAR_30_TOLERANCE_SEC = 0.1

print("MASTER_PATH         :", MASTER_PATH)
print("DURATION_CACHE_PATH :", DURATION_CACHE_PATH)
print("SEGMENT_PATH        :", SEGMENT_PATH)
print("EXCLUSION_PATH      :", EXCLUSION_PATH)

MASTER_PATH         : /Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project/data/metadata/master_manifest_with_split.csv
DURATION_CACHE_PATH : /Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project/data/metadata/audio_duration_cache.csv
SEGMENT_PATH        : /Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project/data/metadata/segment_manifest_10s.csv
EXCLUSION_PATH      : /Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project/data/metadata/segment_exclusions.csv


## 1. Master Manifest 로드

이전 단계에서 확정한 split을 그대로 유지한다.
segment를 생성한 뒤에도 각 segment는 원본 track의 `split`을 상속한다.

In [23]:
master = pd.read_csv(MASTER_PATH)

print("===== MASTER =====")
print("Rows                 :", len(master))
print("Unique original_audio:", master["original_audio"].nunique())
print("REAL                 :", int((master["label"] == "REAL").sum()))
print("FAKE                 :", int((master["label"] == "FAKE").sum()))

print("\nSplit:")
# 같은 원곡의 표본이 여러 분할에 섞이지 않도록 저장된 split을 그대로 사용한다.
print(master["split"].value_counts())

display(master.head())

===== MASTER =====
Rows                 : 3458
Unique original_audio: 296
REAL                 : 296
FAKE                 : 3162

Split:
split
train    2392
test      539
val       527
Name: count, dtype: int64


,sample_id,original_audio,label,label_id,source,genre,generator,audio_path,track_id,description,path_in_dataset,file_exists,split
0,sample_00000,"10,000 People Chanting, ""I'm an Individual"" - ...",REAL,0,FMA,Electronic,NaN,data/raw/FMA/selected_30s/140/140932.mp3,140932.0,NaN,NaN,True,train
1,sample_00001,1984 - Punk Rock Opera,REAL,0,FMA,Rock,NaN,data/raw/FMA/selected_30s/149/149410.mp3,149410.0,NaN,NaN,True,train
2,sample_00002,2 (Wasn't There) - Isle of Pine,REAL,0,FMA,Rock,NaN,data/raw/FMA/selected_30s/066/066449.mp3,66449.0,NaN,NaN,True,val
3,sample_00003,2Much (Andy Spinelli & Alex Sánchez House Edit...,REAL,0,FMA,Electronic,NaN,data/raw/FMA/selected_30s/114/114244.mp3,114244.0,NaN,NaN,True,test
4,sample_00004,3 am West End - statusq,REAL,0,FMA,Electronic,NaN,data/raw/FMA/selected_30s/112/112378.mp3,112378.0,NaN,NaN,True,test


## 2. 실제 오디오 Duration 측정

segment 위치를 정확하게 정하기 위해 실제 파일 duration을 확인한다.

`ffprobe`가 설치되어 있으면 빠르게 duration metadata를 읽는다.
이미 측정한 결과가 `audio_duration_cache.csv`에 있으면 재사용하므로
노트북을 다시 실행해도 모든 파일을 다시 검사하지 않는다.

In [ ]:
# 실제 오디오 Duration 측정
FFPROBE_AVAILABLE = shutil.which("ffprobe") is not None

print("ffprobe available:", FFPROBE_AVAILABLE)

if not FFPROBE_AVAILABLE:
    raise RuntimeError(
        "ffprobe가 필요합니다. Mac 터미널에서 `brew install ffmpeg`를 실행한 뒤 다시 시도하세요."
    )

ffprobe available: True


In [5]:
def probe_duration(path: Path):
    cmd = [
        "ffprobe",
        "-v",
        "error",
        "-show_entries",
        "format=duration",
        "-of",
        "default=noprint_wrappers=1:nokey=1",
        str(path),
    ]

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
    )

    if result.returncode != 0:
        return False, np.nan, result.stderr.strip()

    try:
        return True, float(result.stdout.strip()), ""
    except ValueError:
        return False, np.nan, "duration 값을 float로 변환하지 못함"


if DURATION_CACHE_PATH.exists():
    duration_cache = pd.read_csv(DURATION_CACHE_PATH)
else:
    duration_cache = pd.DataFrame(
        columns=["audio_path", "duration_ok", "actual_duration_sec", "duration_error"]
    )

cached_paths = set(duration_cache["audio_path"].astype(str))

print("Cached durations:", len(duration_cache))
print("Need to probe   :", (~master["audio_path"].astype(str).isin(cached_paths)).sum())

Cached durations: 0
Need to probe   : 3458


In [6]:
new_duration_rows = []

to_probe = master[~master["audio_path"].astype(str).isin(cached_paths)][
    ["audio_path"]
].drop_duplicates()

for i, audio_path in enumerate(to_probe["audio_path"], start=1):
    full_path = PROJECT_ROOT / audio_path

    if not full_path.exists():
        ok = False
        duration = np.nan
        error = "file_not_found"
    else:
        ok, duration, error = probe_duration(full_path)

    new_duration_rows.append(
        {
            "audio_path": audio_path,
            "duration_ok": ok,
            "actual_duration_sec": duration,
            "duration_error": error,
        }
    )

    if i % 250 == 0 or i == len(to_probe):
        print(f"probed: {i}/{len(to_probe)}")

        partial = pd.concat(
            [duration_cache, pd.DataFrame(new_duration_rows)], ignore_index=True
        ).drop_duplicates("audio_path", keep="last")

        partial.to_csv(DURATION_CACHE_PATH, index=False, encoding="utf-8-sig")

if new_duration_rows:
    duration_cache = pd.concat(
        [duration_cache, pd.DataFrame(new_duration_rows)], ignore_index=True
    ).drop_duplicates("audio_path", keep="last")

duration_cache.to_csv(DURATION_CACHE_PATH, index=False, encoding="utf-8-sig")

print("\nDuration cache rows:", len(duration_cache))

probed: 250/3458
probed: 500/3458
probed: 750/3458
probed: 1000/3458
probed: 1250/3458
probed: 1500/3458
probed: 1750/3458
probed: 2000/3458
probed: 2250/3458
probed: 2500/3458
probed: 2750/3458
probed: 3000/3458
probed: 3250/3458
probed: 3458/3458

Duration cache rows: 3458


## 3. Duration 결과를 Master Manifest와 결합

모든 오디오의 duration 측정 성공 여부를 확인한다.

In [8]:
master_duration = master.merge(
    duration_cache[
        ["audio_path", "duration_ok", "actual_duration_sec", "duration_error"]
    ],
    on="audio_path",
    how="left",
    validate="one_to_one",
)

# duration_ok를 확실한 boolean 타입으로 변환
master_duration["duration_ok"] = (
    master_duration["duration_ok"]
    .astype(str)
    .str.strip()
    .str.lower()
    .map(
        {
            "true": True,
            "false": False,
            "1": True,
            "0": False,
        }
    )
    .fillna(False)
    .astype(bool)
)

print("===== DURATION CHECK =====")
print("Success:", int(master_duration["duration_ok"].sum()))
print("Failed :", int((~master_duration["duration_ok"]).sum()))

duration_failures = master_duration[~master_duration["duration_ok"]].copy()

if len(duration_failures):
    display(duration_failures[["sample_id", "label", "audio_path", "duration_error"]])

print("\nActual duration summary:")
display(master_duration["actual_duration_sec"].describe())

===== DURATION CHECK =====
Success: 3458
Failed : 0

Actual duration summary:


count     3458.00
unique    1085.00
top         29.94
freq       293.00
Name: actual_duration_sec, dtype: float64

## 4. 30초 근처 duration 정규화

MP3 codec metadata로 인해 실제 30초 clip이 몇 ms 짧거나 길게 측정될 수 있다.

예를 들어 이전 FMA QC에서는 정상적인 30초 파일이 약
`29.988 ~ 30.015초`로 측정되었다.

따라서 `29.9 ~ 30.1초` 범위의 파일은 segment 계획에서
정확히 30초로 취급한다.

- `actual_duration_sec`: 실제 측정값
- `planning_duration_sec`: segment 위치를 정할 때 사용하는 값

In [9]:
# 30초 근처 duration 정규화
master_duration["planning_duration_sec"] = master_duration["actual_duration_sec"]

near_30 = (
    master_duration["actual_duration_sec"].sub(30.0).abs() <= NEAR_30_TOLERANCE_SEC
)

master_duration.loc[near_30, "planning_duration_sec"] = 30.0

print("Normalized to 30 sec:", int(near_30.sum()))

display(
    master_duration[["label", "actual_duration_sec", "planning_duration_sec"]].head()
)

Normalized to 30 sec: 591


,label,actual_duration_sec,planning_duration_sec
0,REAL,29.988571,30.0
1,REAL,30.014694,30.0
2,REAL,30.014694,30.0
3,REAL,29.988571,30.0
4,REAL,29.988571,30.0


## 5. Track별 Segment 수 결정

planning duration에 따라 다음 규칙을 적용한다.

```text
>= 30초 → 3 segments
>= 20초 → 2 segments
>= 10초 → 1 segment
< 10초  → 0 segments (excluded)
```

In [10]:
# Track별 Segment 수 결정
def planned_segment_count(duration):
    if pd.isna(duration):
        return 0
    if duration >= 30:
        return 3
    if duration >= 20:
        return 2
    if duration >= 10:
        return 1
    return 0


master_duration["planned_segments"] = master_duration["planning_duration_sec"].apply(
    planned_segment_count
)

print("===== PLANNED SEGMENTS PER TRACK =====")
print(master_duration["planned_segments"].value_counts().sort_index())

print("\nBy label:")
display(pd.crosstab(master_duration["label"], master_duration["planned_segments"]))

===== PLANNED SEGMENTS PER TRACK =====
planned_segments
1       2
2     293
3    3163
Name: count, dtype: int64

By label:


planned_segments,1,2,3
label,,,
FAKE,2,293,2867
REAL,0,0,296


## 6. Segment 위치 생성

### 3개 segment
- start: `0 ~ 10초`
- middle: `(duration - 10) / 2`부터 10초
- end: 마지막 10초

### 2개 segment
- start: `0 ~ 10초`
- end: 마지막 10초

### 1개 segment
- center: 중앙 10초

30초로 정규화된 MP3가 실제로 수 ms 짧은 경우에는
마지막 segment가 실제 파일 끝을 아주 조금 넘을 수 있다.

이 경우 `requires_padding=True`로 표시하고,
향후 실제 waveform을 읽을 때 부족한 부분만 zero-padding한다.

In [11]:
# 1개 segment
def build_segments(row):
    d = row["planning_duration_sec"]

    if pd.isna(d) or d < 10:
        return []

    if d >= 30:
        specs = [
            ("start", 0.0),
            ("middle", (d - SEGMENT_SEC) / 2.0),
            ("end", d - SEGMENT_SEC),
        ]
    elif d >= 20:
        specs = [
            ("start", 0.0),
            ("end", d - SEGMENT_SEC),
        ]
    else:
        specs = [
            ("center", (d - SEGMENT_SEC) / 2.0),
        ]

    result = []

    for segment_index, (role, start_sec) in enumerate(specs):
        start_sec = round(max(0.0, float(start_sec)), 6)
        end_sec = round(start_sec + SEGMENT_SEC, 6)

        actual_d = row["actual_duration_sec"]

        available_sec = max(0.0, min(SEGMENT_SEC, float(actual_d) - start_sec))

        pad_sec = max(0.0, SEGMENT_SEC - available_sec)

        result.append(
            {
                "segment_index": segment_index,
                "segment_role": role,
                "start_sec": start_sec,
                "end_sec": end_sec,
                "segment_duration_sec": SEGMENT_SEC,
                "available_audio_sec": round(available_sec, 6),
                "pad_sec": round(pad_sec, 6),
                "requires_padding": pad_sec > 1e-6,
            }
        )

    return result

In [12]:
# 1개 segment
segment_rows = []
exclusion_rows = []

for _, row in master_duration.iterrows():
    segments = build_segments(row)

    if not segments:
        exclusion_rows.append(
            {
                "sample_id": row["sample_id"],
                "original_audio": row["original_audio"],
                "label": row["label"],
                "genre": row["genre"],
                "generator": row["generator"],
                "split": row["split"],
                "audio_path": row["audio_path"],
                "actual_duration_sec": row["actual_duration_sec"],
                "reason": (
                    "duration_probe_failed"
                    if not bool(row["duration_ok"])
                    else "duration_under_10_sec"
                ),
            }
        )
        continue

    for seg in segments:
        segment_rows.append(
            {
                "track_sample_id": row["sample_id"],
                "original_audio": row["original_audio"],
                "label": row["label"],
                "label_id": row["label_id"],
                "source": row["source"],
                "genre": row["genre"],
                "generator": row["generator"],
                "audio_path": row["audio_path"],
                "track_id": row["track_id"],
                "split": row["split"],
                "actual_duration_sec": row["actual_duration_sec"],
                "planning_duration_sec": row["planning_duration_sec"],
                "planned_segments": row["planned_segments"],
                **seg,
            }
        )

segment_manifest = pd.DataFrame(segment_rows)

segment_manifest.insert(
    0, "segment_id", [f"seg_{i:06d}" for i in range(len(segment_manifest))]
)

segment_exclusions = pd.DataFrame(exclusion_rows)

print("Segment rows :", len(segment_manifest))
print("Excluded tracks:", len(segment_exclusions))

display(segment_manifest.head(10))

Segment rows : 10077
Excluded tracks: 0


,segment_id,track_sample_id,original_audio,label,label_id,source,genre,generator,audio_path,track_id,...,planning_duration_sec,planned_segments,segment_index,segment_role,start_sec,end_sec,segment_duration_sec,available_audio_sec,pad_sec,requires_padding
0,seg_000000,sample_00000,"10,000 People Chanting, ""I'm an Individual"" - ...",REAL,0,FMA,Electronic,NaN,data/raw/FMA/selected_30s/140/140932.mp3,140932.0,...,30.0,3,0,start,0.0,10.0,10.0,10.000000,0.000000,False
1,seg_000001,sample_00000,"10,000 People Chanting, ""I'm an Individual"" - ...",REAL,0,FMA,Electronic,NaN,data/raw/FMA/selected_30s/140/140932.mp3,140932.0,...,30.0,3,1,middle,10.0,20.0,10.0,10.000000,0.000000,False
2,seg_000002,sample_00000,"10,000 People Chanting, ""I'm an Individual"" - ...",REAL,0,FMA,Electronic,NaN,data/raw/FMA/selected_30s/140/140932.mp3,140932.0,...,30.0,3,2,end,20.0,30.0,10.0,9.988571,0.011429,True
3,seg_000003,sample_00001,1984 - Punk Rock Opera,REAL,0,FMA,Rock,NaN,data/raw/FMA/selected_30s/149/149410.mp3,149410.0,...,30.0,3,0,start,0.0,10.0,10.0,10.000000,0.000000,False
4,seg_000004,sample_00001,1984 - Punk Rock Opera,REAL,0,FMA,Rock,NaN,data/raw/FMA/selected_30s/149/149410.mp3,149410.0,...,30.0,3,1,middle,10.0,20.0,10.0,10.000000,0.000000,False
5,seg_000005,sample_00001,1984 - Punk Rock Opera,REAL,0,FMA,Rock,NaN,data/raw/FMA/selected_30s/149/149410.mp3,149410.0,...,30.0,3,2,end,20.0,30.0,10.0,10.000000,0.000000,False
6,seg_000006,sample_00002,2 (Wasn't There) - Isle of Pine,REAL,0,FMA,Rock,NaN,data/raw/FMA/selected_30s/066/066449.mp3,66449.0,...,30.0,3,0,start,0.0,10.0,10.0,10.000000,0.000000,False
7,seg_000007,sample_00002,2 (Wasn't There) - Isle of Pine,REAL,0,FMA,Rock,NaN,data/raw/FMA/selected_30s/066/066449.mp3,66449.0,...,30.0,3,1,middle,10.0,20.0,10.0,10.000000,0.000000,False
8,seg_000008,sample_00002,2 (Wasn't There) - Isle of Pine,REAL,0,FMA,Rock,NaN,data/raw/FMA/selected_30s/066/066449.mp3,66449.0,...,30.0,3,2,end,20.0,30.0,10.0,10.000000,0.000000,False
9,seg_000009,sample_00003,2Much (Andy Spinelli & Alex Sánchez House Edit...,REAL,0,FMA,Electronic,NaN,data/raw/FMA/selected_30s/114/114244.mp3,114244.0,...,30.0,3,0,start,0.0,10.0,10.0,10.000000,0.000000,False


## 7. Segment 수 및 역할 분포 확인

In [13]:
print("===== SEGMENTS PER TRACK =====")
track_segment_counts = (
    # 같은 곡의 10초 구간을 묶어 곡 단위 결과를 만든다.
    segment_manifest.groupby("track_sample_id").size().value_counts().sort_index()
)

print(track_segment_counts)

print("\n===== SEGMENT ROLE =====")
print(segment_manifest["segment_role"].value_counts())

print("\n===== BY LABEL =====")
display(pd.crosstab(segment_manifest["label"], segment_manifest["segment_role"]))

===== SEGMENTS PER TRACK =====
1       2
2     293
3    3163
Name: count, dtype: int64

===== SEGMENT ROLE =====
segment_role
start     3456
end       3456
middle    3163
center       2
Name: count, dtype: int64

===== BY LABEL =====


segment_role,center,end,middle,start
label,,,,
FAKE,2,3160,2867,3160
REAL,0,296,296,296


## 8. Padding 필요 여부 확인

대부분의 segment는 정확히 10초의 실제 audio를 갖는다.

단, codec duration 오차로 실제 30초 MP3가 수 ms 짧을 경우
마지막 segment만 아주 작은 padding이 필요할 수 있다.

In [14]:
# Padding 필요 여부 확인
padding_segments = segment_manifest[segment_manifest["requires_padding"]].copy()

print("Requires padding:", len(padding_segments))

if len(padding_segments):
    print("\nPadding seconds summary:")
    display(padding_segments["pad_sec"].describe())

    print("\nLargest padding examples:")
    display(
        padding_segments[
            [
                "segment_id",
                "track_sample_id",
                "label",
                "audio_path",
                "segment_role",
                "start_sec",
                "actual_duration_sec",
                "pad_sec",
            ]
        ]
        .sort_values("pad_sec", ascending=False)
        .head(20)
    )

Requires padding: 463

Padding seconds summary:


count    463.000000
mean       0.042166
std        0.023438
min        0.011429
25%        0.011429
50%        0.060000
75%        0.060000
max        0.060000
Name: pad_sec, dtype: float64


Largest padding examples:


,segment_id,track_sample_id,label,audio_path,segment_role,start_sec,actual_duration_sec,pad_sec
9383,seg_009383,sample_03226,FAKE,data/raw/Echoes/Echoes/TTA/musicgen/End_of_the...,end,20.0,29.94,0.06
9644,seg_009644,sample_03313,FAKE,data/raw/Echoes/Echoes/TTA/musicgen/Motocross_...,end,20.0,29.94,0.06
9638,seg_009638,sample_03311,FAKE,data/raw/Echoes/Echoes/TTA/musicgen/Morning_Be...,end,20.0,29.94,0.06
9635,seg_009635,sample_03310,FAKE,data/raw/Echoes/Echoes/TTA/musicgen/Moontime_T...,end,20.0,29.94,0.06
9632,seg_009632,sample_03309,FAKE,data/raw/Echoes/Echoes/TTA/musicgen/Monkeystag...,end,20.0,29.94,0.06
9629,seg_009629,sample_03308,FAKE,data/raw/Echoes/Echoes/TTA/musicgen/Moja_Krypt...,end,20.0,29.94,0.06
9626,seg_009626,sample_03307,FAKE,data/raw/Echoes/Echoes/TTA/musicgen/Millitary_...,end,20.0,29.94,0.06
9623,seg_009623,sample_03306,FAKE,data/raw/Echoes/Echoes/TTA/musicgen/Mettle_Pip...,end,20.0,29.94,0.06
9620,seg_009620,sample_03305,FAKE,data/raw/Echoes/Echoes/TTA/musicgen/mc_trabale...,end,20.0,29.94,0.06
9617,seg_009617,sample_03304,FAKE,data/raw/Echoes/Echoes/TTA/musicgen/March_of_t...,end,20.0,29.94,0.06


## 9. Segment Boundary 검증

각 segment에 대해 다음을 확인한다.

- `start_sec >= 0`
- segment 목표 길이 = 10초
- 실제 audio에서 확보 가능한 구간 + padding = 약 10초
- 같은 track에서 segment 시작점이 중복되지 않음

In [15]:
# Segment Boundary 검증
invalid_start = segment_manifest[segment_manifest["start_sec"] < 0]

invalid_target_duration = segment_manifest[
    ~np.isclose(segment_manifest["segment_duration_sec"], SEGMENT_SEC, atol=1e-6)
]

invalid_available = segment_manifest[
    (segment_manifest["available_audio_sec"] + segment_manifest["pad_sec"])
    .sub(SEGMENT_SEC)
    .abs()
    > 1e-4
]

duplicate_positions = segment_manifest.duplicated(
    subset=["track_sample_id", "start_sec"], keep=False
)

print("Invalid start          :", len(invalid_start))
print("Invalid target duration:", len(invalid_target_duration))
print("Invalid available+pad  :", len(invalid_available))
print("Duplicate positions    :", int(duplicate_positions.sum()))

Invalid start          : 0
Invalid target duration: 0
Invalid available+pad  : 0
Duplicate positions    : 0


## 10. Split Leakage 재검증

segment를 생성해도 기존 `original_audio` group split이 유지되어야 한다.

같은 `original_audio`가 여러 split에 존재하는지 다시 검사한다.

In [16]:
segment_group_split_count = segment_manifest.groupby("original_audio")[
    "split"
].nunique()

leaked_groups = segment_group_split_count[segment_group_split_count > 1]

print(
    "Original_audio groups in segments:", segment_manifest["original_audio"].nunique()
)

print("Groups appearing in >1 split:", len(leaked_groups))

print("\nSegment split distribution:")
# 같은 원곡의 표본이 여러 분할에 섞이지 않도록 저장된 split을 그대로 사용한다.
print(segment_manifest["split"].value_counts())

Original_audio groups in segments: 296
Groups appearing in >1 split: 0

Segment split distribution:
split
train    6967
test     1572
val      1538
Name: count, dtype: int64


## 11. REAL / FAKE Segment 분포 확인

트랙 단계의 클래스 불균형이 segment 단계에서 어떻게 변하는지 확인한다.

In [17]:
# REAL / FAKE Segment 분포 확인
segment_label_count = pd.crosstab(segment_manifest["split"], segment_manifest["label"])

segment_label_ratio = pd.crosstab(
    segment_manifest["split"], segment_manifest["label"], normalize="index"
).round(4)

print("===== SEGMENT LABEL COUNT =====")
display(segment_label_count)

print("===== SEGMENT LABEL RATIO =====")
display(segment_label_ratio)

===== SEGMENT LABEL COUNT =====


label,FAKE,REAL
split,,
test,1437,135
train,6346,621
val,1406,132


===== SEGMENT LABEL RATIO =====


label,FAKE,REAL
split,,
test,0.9141,0.0859
train,0.9109,0.0891
val,0.9142,0.0858


## 12. Generator별 Segment 수 확인

FAKE만 대상으로 generator별 segment 수를 확인한다.

In [18]:
# Generator별 Segment 수 확인
fake_segments = segment_manifest[segment_manifest["label"] == "FAKE"]

generator_segment_count = (
    fake_segments["generator"].value_counts().rename("segments").to_frame()
)

display(generator_segment_count)

,segments
generator,
suno,900
udio,900
elevenlabs,900
diffrhythm,897
brev,894
acestep,882
musicgen,879
audioldm,876
songgen,582


## 13. Exclusion 확인

10초 미만 또는 duration probe 실패로 segment를 만들지 못한 track이 있다면 확인한다.

제외가 발생했다고 해서 즉시 오류는 아니지만,
학습 데이터에서 빠지는 sample이므로 반드시 기록해야 한다.

In [19]:
# Exclusion 확인
print("Excluded tracks:", len(segment_exclusions))

if len(segment_exclusions):
    print("\nReason:")
    print(segment_exclusions["reason"].value_counts())

    display(segment_exclusions.head(30))
else:
    print("No exclusions.")

Excluded tracks: 0
No exclusions.


## 14. 최종 Segment QC

핵심 조건:

- 모든 segment 목표 길이 = 10초
- segment ID 중복 없음
- source-family leakage 없음
- split 누락 없음
- audio path 누락 없음
- segment 경계 계산 오류 없음

In [20]:
# 최종 Segment QC
qc_summary = pd.DataFrame(
    {
        "check": [
            "source_tracks",
            "segment_rows",
            "tracks_with_segments",
            "excluded_tracks",
            "unique_segment_id",
            "duplicate_segment_id",
            "missing_split",
            "missing_audio_path",
            "leaked_original_audio_groups",
            "invalid_start",
            "invalid_target_duration",
            "invalid_available_plus_padding",
            "duplicate_segment_positions",
            "padding_segments",
        ],
        "value": [
            len(master_duration),
            len(segment_manifest),
            segment_manifest["track_sample_id"].nunique(),
            len(segment_exclusions),
            segment_manifest["segment_id"].nunique(),
            int(segment_manifest["segment_id"].duplicated().sum()),
            int(segment_manifest["split"].isna().sum()),
            int(segment_manifest["audio_path"].isna().sum()),
            len(leaked_groups),
            len(invalid_start),
            len(invalid_target_duration),
            len(invalid_available),
            int(duplicate_positions.sum()),
            len(padding_segments),
        ],
    }
)

display(qc_summary)

core_qc_pass = (
    segment_manifest["segment_id"].duplicated().sum() == 0
    and segment_manifest["split"].isna().sum() == 0
    and segment_manifest["audio_path"].isna().sum() == 0
    and len(leaked_groups) == 0
    and len(invalid_start) == 0
    and len(invalid_target_duration) == 0
    and len(invalid_available) == 0
    and int(duplicate_positions.sum()) == 0
)

print("===== FINAL RESULT =====")
print("Segment Core QC PASS:", core_qc_pass)

,check,value
0,source_tracks,3458
1,segment_rows,10077
2,tracks_with_segments,3458
3,excluded_tracks,0
4,unique_segment_id,10077
5,duplicate_segment_id,0
6,missing_split,0
7,missing_audio_path,0
8,leaked_original_audio_groups,0
9,invalid_start,0


===== FINAL RESULT =====
Segment Core QC PASS: True


## 15. Segment Manifest 저장

다음 파일을 저장한다.

1. `segment_manifest_10s.csv`
   - 모델 학습에서 사용할 10초 segment 정의

2. `segment_exclusions.csv`
   - 10초 미만 또는 duration 측정 실패로 제외된 track 기록

3. `audio_duration_cache.csv`
   - 실제 오디오 duration 측정 결과

In [21]:
if not core_qc_pass:
    raise RuntimeError(
        "Segment Core QC가 통과하지 않았습니다. 저장 전에 위 결과를 확인하세요."
    )

SEGMENT_PATH.parent.mkdir(parents=True, exist_ok=True)

segment_manifest.to_csv(SEGMENT_PATH, index=False, encoding="utf-8-sig")

segment_exclusions.to_csv(EXCLUSION_PATH, index=False, encoding="utf-8-sig")

print("Saved segment manifest :", SEGMENT_PATH)
print("Saved exclusions       :", EXCLUSION_PATH)
print("Saved duration cache    :", DURATION_CACHE_PATH)
print("Segments                :", len(segment_manifest))

Saved segment manifest : /Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project/data/metadata/segment_manifest_10s.csv
Saved exclusions       : /Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project/data/metadata/segment_exclusions.csv
Saved duration cache    : /Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project/data/metadata/audio_duration_cache.csv
Segments                : 10077


## 이어지는 기록

저장된 Segment 위치는 08번의 음향 특징 추출에 사용한다.

## 구간 목록

- 3,458개 track에서 **10,077개 10초 segment**를 생성했다.
- Segment split은 Train 6,967, Validation 1,538, Test 1,572개다.
- Label 분포는 FAKE 9,189개, REAL 888개이며 제외된 track은 **0개**다.
- 모든 segment가 원본 track의 `original_audio` 기반 split을 그대로 상속한다.
- Manifest, exclusion report, duration cache 저장을 완료했다.